# Sesión 3.01 · Pinecone Cloud mediante SDK nativo

Pinecone será la principal alternativa gestionada de esta sesión. No tendremos que administrar nodos, discos ni procesos del servidor, pero eso no elimina nuestras responsabilidades. Seguiremos decidiendo cómo se organiza el esquema, qué datos pertenecen a cada namespace, cómo se controlan las cuotas, qué visibilidad tienen las escrituras y cómo se eliminan los registros de forma segura.

El notebook mantendrá exactamente el protocolo común definido en la sesión anterior. Utilizará los embeddings ya calculados, no delegará la inferencia en Pinecone y trabajará directamente con el SDK nativo, sin introducir todavía una capa de abstracción como LangChain.

Esta separación es deliberada. Si el ranking difiere del oráculo exacto, queremos poder atribuir la diferencia al contrato real de Pinecone: la configuración del índice, el filtrado, la consistencia, la semántica del score o el comportamiento del cliente. Cambiar también el encoder o añadir otra librería haría más difícil localizar el origen.

<a id="s03-pinecone-indice"></a>

## Índice de contenidos

1. [Configuración del entorno y del índice](#s03-pinecone-configuracion)
2. [Ingesta](#s03-pinecone-ingesta)
3. [Verificación de la carga](#s03-pinecone-verificacion)
4. [Resultados de búsqueda](#s03-pinecone-resultados)
5. [Búsqueda sin filtro](#s03-pinecone-busqueda-sin-filtro)
6. [Búsqueda filtrada](#s03-pinecone-busqueda-filtrada)
7. [Operaciones CRUD](#s03-pinecone-operaciones-crud)
8. [Informe de ejecución](#s03-pinecone-informe)
9. [Consideraciones específicas](#s03-pinecone-consideraciones)
10. [Próximos pasos](#s03-pinecone-proximos-pasos)
11. [Limpieza](#s03-pinecone-limpieza)

## Objetivo del laboratorio

También registraremos tiempos, pero solo para describir esta ejecución concreta. No los utilizaremos para comparar proveedores, porque una llamada a un servicio gestionado y una ejecución local no comparten red, hardware, caché ni condiciones de carga.

El recorrido comenzará con un **preflight**. Si faltan credenciales, el índice no existe o su configuración no coincide con el contrato esperado, el notebook se detendrá y mostrará la causa. Continuar después de un fallo con una variable vacía como `results = []` ocultaría información importante: no podríamos distinguir entre una búsqueda que realmente no encontró vecinos y una consulta que nunca llegó a ejecutarse.

In [ ]:
from pathlib import Path
import json
import os
import platform
import time

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

from vector_database_session import (
    ProviderRun,
    SearchHit,
    evaluate_run,
    exact_top_k,
    iter_record_batches,
    load_session_data,
    record_id_for_product,
    validate_resource_name,
    wait_until,
    write_provider_run,
)

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
load_dotenv(PROJECT_ROOT / ".env")
data = load_session_data(memory_map=True)
TELEVISOR_QUERY_ID = "semantic-101352"
TALADRO_QUERY_ID = "semantic-100455"
TOP_K = 10

In [ ]:
RESOURCE_NAME = validate_resource_name(os.getenv("PINECONE_INDEX_NAME", "bbdd-vectoriales-s03-products"))
BATCH_SIZE = 500
NAMESPACE = os.getenv("PINECONE_NAMESPACE", "esci-es-s03")
TEMPORARY_TEST_ID = record_id_for_product("S03-TEMPORARY-TEST")
assert os.getenv("S03_ALLOW_REMOTE_CLEANUP", "false").lower() != "true", "La limpieza remota no pertenece a la ejecución docente"
print({"python": platform.python_version(), "resource": RESOURCE_NAME, "batch_size": BATCH_SIZE})

<a id="s03-pinecone-configuracion"></a>

## 1. Preflight, conexión y esquema

El índice serverless pertenece al plano de control: es el recurso que Pinecone crea y configura para almacenar los vectores. Dentro de él, el namespace actúa como la frontera lógica de este experimento y nos permite aislar sus datos sin necesitar un índice independiente.

En este modo no configuramos directamente el algoritmo ANN ni parámetros internos como `M`, `efSearch` o `nprobe`. Pinecone expone un contrato de servicio y se encarga de la estructura de búsqueda subyacente. Nuestra responsabilidad consiste en definir correctamente aquello que sí forma parte del esquema observable: la dimensión, la métrica y el namespace utilizado.

La creación será condicional. Si el índice ya existe, lo reutilizaremos y comprobaremos que acepta vectores de 384 dimensiones y utiliza similitud coseno. Si no existe, lo crearemos con esa configuración.

No eliminaremos ni recrearemos el recurso simplemente para conseguir que una celda continúe. Si el índice existente tiene otra dimensión o métrica, el notebook debe detenerse y mostrar la incompatibilidad. Borrar el recurso ocultaría el error, eliminaría evidencia útil y podría destruir datos que no pertenecen a esta ejecución.

In [ ]:
from pinecone import Pinecone, ServerlessSpec, __version__ as provider_version

api_key = os.getenv("PINECONE_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("PINECONE_API_KEY está vacía; no existe un resultado vacío atribuible al buscador")

client = Pinecone(api_key=api_key)

cloud = os.getenv("PINECONE_CLOUD", "aws")
region = os.getenv("PINECONE_REGION", "us-east-1")

if not client.has_index(RESOURCE_NAME):
    client.create_index(
        name=RESOURCE_NAME,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud=cloud, region=region),
        # deletion_protection="enabled",  # Descomenta esta línea si quieres activar la protección frente a un borrado accidental del índice
        tags={"course": "bbdd-vectoriales", "session": "03"},
    )
_, ready_seconds, ready_attempts = wait_until(
    lambda: client.describe_index(RESOURCE_NAME),
    accept=lambda description: bool(description.status["ready"]),
    timeout_seconds=300,
    interval_seconds=2,
    description="Pinecone index readiness",
)

description = client.describe_index(RESOURCE_NAME)

assert description.dimension == 384 and str(description.metric).lower().endswith("cosine"), description

index = client.Index(RESOURCE_NAME)
print({"version": provider_version, "host": index.host, "ready_s": ready_seconds, "attempts": ready_attempts})

Antes de ingerir ningún vector, conviene inspeccionar el recurso desde la consola o mediante el endpoint administrativo. La primera pregunta es si la dimensión y la métrica forman parte de un esquema inmutable: si están fijadas al crear el índice, cualquier cambio exigirá una migración hacia un recurso nuevo.

También debemos comprobar cómo trata Pinecone los metadatos. El filtro `brand == "Einhell"` puede aplicarse sobre la metadata almacenada, pero interesa saber si requiere alguna configuración adicional o si el servicio mantiene internamente las estructuras necesarias para filtrarla.

Esta inspección permite separar dos tipos de cambios. Modificar datos, añadir registros o actualizar metadatos puede hacerse normalmente en caliente. Cambiar la dimensión, la métrica o cualquier propiedad estructural del índice suele implicar crear una nueva versión, volver a ingerir y cambiar después el tráfico.

La propia sintaxis de creación ya ofrece pistas sobre el modelo operativo del proveedor: qué decisiones quedan fijadas desde el principio, cuáles pueden ajustarse durante la vida del recurso y cuáles obligan a una migración.

<a id="s03-pinecone-ingesta"></a>

## 2. Ingesta idempotente de 50.000 registros

Cada producto utilizará el mismo UUIDv5 en los cinco motores. Como el identificador se genera de forma determinista a partir de `product_id`, volver a ejecutar la ingesta produce exactamente los mismos IDs.

La operación `upsert` aprovecha esa propiedad: si el registro no existe, lo inserta; si ya estaba presente, lo reemplaza o actualiza según la semántica del motor. Gracias a ello, el notebook puede reanudarse después de una interrupción sin crear duplicados.

Esto no significa que toda la tubería ofrezca una garantía de *exactly once*. Si el proceso falla a mitad de la carga, algunos batches pueden haber sido confirmados y otros no. La idempotencia no evita ese estado parcial, pero hace que repetir la operación sea seguro: los lotes ya escritos se procesarán de nuevo sobre los mismos IDs y los pendientes podrán completarse.

Utilizaremos batches de 500 registros como punto de partida para la práctica. No debe interpretarse como un tamaño óptimo universal. En producción, el valor adecuado depende de los límites de cada petición, la memoria disponible en el cliente, el tamaño de los metadatos, la latencia de red y el grado de concurrencia.

Registraremos el tiempo total de ingesta para describir esta ejecución y detectar posibles anomalías. No lo utilizaremos para comparar Pinecone con motores ejecutados en otros entornos, porque las condiciones de hardware, red y despliegue no son equivalentes.

In [ ]:
ingestion_started = time.perf_counter()

for batch_number, records in enumerate(iter_record_batches(data, batch_size=BATCH_SIZE), start=1):
    vectors = [
        {"id": record.record_id, "values": record.embedding, "metadata": {**record.flat_metadata(), "text": record.text}}
        for record in records
    ]
    index.upsert(vectors=vectors, namespace=NAMESPACE)
    if batch_number % 20 == 0:
        print(f"{batch_number * BATCH_SIZE:,} registros enviados")
        
ingestion_ms = (time.perf_counter() - ingestion_started) * 1000

<a id="s03-pinecone-verificacion"></a>

## 3. Recuento y estado de indexación

Que el servidor haya aceptado 50.000 escrituras no significa todavía que los 50.000 registros estén disponibles para búsqueda. La ingesta y la visibilidad forman parte de momentos distintos del ciclo de escritura.

Por eso no daremos por completada la carga a partir del número de batches enviados ni de las respuestas correctas del cliente. Consultaremos el estado que expone el propio motor y comprobaremos cuántos registros reconoce dentro del namespace.

En un servicio con visibilidad eventual, ese recuento puede tardar unos instantes en alcanzar el valor esperado. Esperaremos de forma acotada y registraremos la evolución observada. Si se agota el plazo sin llegar a 50.000 registros visibles, el notebook fallará mostrando el último estado recibido.

De este modo distinguiremos entre tres situaciones diferentes: una escritura rechazada, una escritura aceptada pero todavía no visible y una carga completamente disponible para consulta.

In [ ]:
stats, visibility_seconds, visibility_attempts = wait_until(
    lambda: index.describe_index_stats(),
    accept=lambda current: current.namespaces.get(NAMESPACE, {}).get("vector_count", 0) == 50_000,
    timeout_seconds=300,
    interval_seconds=2,
    description="50.000 vectores visibles en el namespace",
)
record_count = int(stats.namespaces[NAMESPACE]["vector_count"])

assert record_count == 50_000, f'Recuento inesperado: {record_count}'
print({'record_count': record_count, 'ingestion_ms': ingestion_ms, 'visible_s': visibility_seconds})

<a id="s03-pinecone-resultados"></a>

## 4. Normalizar la respuesta sin borrar su semántica

Cada motor devuelve sus resultados con una estructura propia. Para poder comparar rankings y mostrar una tabla común, traduciremos esas respuestas a un modelo compartido llamado `SearchHit`.

Ese modelo conservará el ID recuperado, pero también tres campos necesarios para interpretar correctamente la puntuación: `native_score`, `score_kind` y `higher_is_better`. Así sabremos si el proveedor devuelve una similitud o una distancia y en qué sentido debe ordenarse.

La normalización no convierte esas puntuaciones en magnitudes equivalentes. Una distancia de 0,2 y una similitud de 0,8 pueden inducir el mismo ranking, pero no representan la misma cantidad ni deben compararse directamente entre motores.

El objetivo es más sencillo: evitar repetir cinco veces el código de evaluación y trabajar con una interfaz común sin ocultar la semántica original de cada proveedor.

In [ ]:
def native_search(query_vector, *, k=10, brand=None):
    metadata_filter = {"brand": {"$eq": brand}} if brand else None
    response = index.query(
        namespace=NAMESPACE,
        vector=np.asarray(query_vector, dtype=np.float32).tolist(),
        top_k=k,
        filter=metadata_filter,
        include_metadata=True,
    )
    return [
        SearchHit(
            record_id=match.id,
            product_id=match.metadata["product_id"],
            vector_id=int(match.metadata["vector_id"]),
            title=match.metadata["title"],
            brand=match.metadata.get("brand", ""),
            native_score=float(match.score),
            score_kind="similarity",
            higher_is_better=True,
            rank=rank,
        )
        for rank, match in enumerate(response.matches, start=1)
    ]

<a id="s03-pinecone-busqueda-sin-filtro"></a>

## 5. El televisor: fidelidad no equivale a relevancia

Volveremos ahora a la consulta problemática del televisor. Antes de inspeccionar la tabla, conviene formular una predicción: si Pinecone reproduce fielmente el espacio generado por E5, debería devolver en primera posición el mismo producto que el oráculo exacto.

En este caso, ese producto es un mantel. El resultado es claramente poco útil para la intención de búsqueda, pero su presencia no demuestra un fallo de la base de datos. Al contrario: si el motor devuelve el mismo UUID y mantiene un `recall@10` alto frente al oráculo, estará reproduciendo correctamente una geometría semántica que ya contenía ese error.

Que cinco bases de datos distintas recuperen el mismo mantel no lo convierte en relevante. Lo convierte en una evidencia reproducible de que el problema se encuentra antes, en el encoder, en la representación del texto o en los datos con los que se construyó el espacio.

Esta distinción será central durante la inspección. La fidelidad mide cuánto respeta el motor el ranking definido por los vectores; la relevancia mide si ese ranking responde realmente a la necesidad del usuario. Ambas propiedades pueden coincidir, pero no son equivalentes.

In [ ]:
television_row = data.query_row(TELEVISOR_QUERY_ID)
television_vector = data.query_vector(TELEVISOR_QUERY_ID)

exact_hits = exact_top_k(data, television_vector, k=TOP_K)

query_started = time.perf_counter()
native_hits = native_search(television_vector, k=TOP_K)
query_ms = (time.perf_counter() - query_started) * 1000

television_evaluation = evaluate_run(exact_hits, native_hits, k=TOP_K)

display(Markdown(f"**Consulta:** {television_row['query_text']}"))
display(pd.DataFrame([hit.as_dict() for hit in native_hits]))
television_evaluation

Interpreta `recall@10` como una medida de fidelidad algorítmica frente al oráculo exacto, no como una medida de relevancia humana. Un valor inferior a uno indica que el motor no ha reproducido por completo el top-10 de fuerza bruta, pero no identifica por sí solo la causa. La pérdida puede proceder del índice ANN, de una configuración de búsqueda demasiado restrictiva o de que algunos registros todavía no sean visibles para la consulta.

Un `recall@10` igual a uno cuenta una historia distinta. Si el ranking coincide con el oráculo y el mantel sigue apareciendo en primera posición, la base de datos ha reproducido correctamente el espacio de E5. En ese caso, el error debe atribuirse a la representación o al modelo, no al mecanismo de recuperación.

<a id="s03-pinecone-busqueda-filtrada"></a>

## 6. El taladro: búsqueda global y búsqueda condicionada

Ejecutaremos la misma consulta sobre taladros de dos formas. La primera buscará los vecinos más próximos dentro de todo el catálogo. La segunda añadirá la condición `brand == "Einhell"` directamente a la petición enviada al motor.

Esto significa que el top-$k$ filtrado debe calcularse sobre el conjunto de productos Einhell, no sobre los diez primeros resultados de la búsqueda global. No recuperaremos primero diez vecinos y eliminaremos después los que pertenezcan a otras marcas, porque ese procedimiento podría devolver menos resultados y perder candidatos válidos situados más abajo en el ranking general.

La comparación permitirá observar cómo cambia el universo de búsqueda cuando el filtro forma parte del contrato del motor. También compararemos los IDs devueltos con un oráculo exacto construido sobre el mismo subconjunto de productos Einhell, de modo que podamos medir la fidelidad del ranking condicionado y no solo comprobar que todos los resultados pertenecen a la marca solicitada.

In [ ]:
drill_row = data.query_row(TALADRO_QUERY_ID)
drill_vector = data.query_vector(TALADRO_QUERY_ID)

global_drill_hits = native_search(drill_vector, k=TOP_K)
filtered_drill_hits = native_search(drill_vector, k=TOP_K, brand="Einhell")

exact_filtered_hits = exact_top_k(data, drill_vector, k=TOP_K, brand="Einhell")

filtered_evaluation = evaluate_run(exact_filtered_hits, filtered_drill_hits, k=TOP_K)
assert filtered_drill_hits and all(hit.brand == "Einhell" for hit in filtered_drill_hits)
display(Markdown(f"**Consulta:** {drill_row['query_text']}"))
display(pd.DataFrame([hit.as_dict() for hit in global_drill_hits]))
display(pd.DataFrame([hit.as_dict() for hit in filtered_drill_hits]))
filtered_evaluation

<a id="s03-pinecone-operaciones-crud"></a>

## 7. Canary Test: `upsert → fetch/query → update → delete`

El canary test utilizará un UUID conocido y reservado para esta sesión. Sobre ese único registro recorreremos el ciclo completo: lo insertaremos, comprobaremos que puede recuperarse por ID y mediante búsqueda, modificaremos uno de sus campos y, finalmente, lo eliminaremos.

El objetivo no es solo confirmar que Pinecone admite operaciones CRUD. También queremos observar cuánto tarda cada cambio en hacerse visible desde las distintas rutas de lectura. Por eso registraremos los intentos y el tiempo transcurrido hasta detectar la inserción, la actualización y el borrado.

Al terminar eliminaremos únicamente el registro temporal de prueba. No borraremos el namespace ni el índice completo, porque la prueba debe ser segura y no afectar al resto de los datos ingeridos.

Un resultado inmediato tampoco demuestra que el sistema sea fuertemente consistente en todos los casos. Solo describe lo ocurrido para esta operación, en esta ruta de lectura y durante esta ejecución concreta. La prueba aporta evidencia observable, pero no permite generalizar una garantía más amplia que la documentada por el proveedor.

In [ ]:
temporary_test_vector = television_vector.tolist()
temporary_test_metadata = {"record_id": TEMPORARY_TEST_ID, "product_id": "S03-TEMPORARY-TEST", "vector_id": -1, "title": "Registro temporal de prueba", "brand": "S03", "color": "amarillo", "locale": "es", "text": "registro temporal"}

started = time.perf_counter()

index.upsert(vectors=[{"id": TEMPORARY_TEST_ID, "values": temporary_test_vector, "metadata": temporary_test_metadata}], namespace=NAMESPACE)
fetched, upsert_visible_s, upsert_attempts = wait_until(
    lambda: index.fetch(ids=[TEMPORARY_TEST_ID], namespace=NAMESPACE),
    accept=lambda response: TEMPORARY_TEST_ID in response.vectors,
    timeout_seconds=60,
    description="registro temporal de prueba de Pinecone tras el upsert",
)

index.update(id=TEMPORARY_TEST_ID, set_metadata={"brand": "S03-updated"}, namespace=NAMESPACE)
_, update_visible_s, update_attempts = wait_until(
    lambda: index.fetch(ids=[TEMPORARY_TEST_ID], namespace=NAMESPACE),
    accept=lambda response: response.vectors.get(TEMPORARY_TEST_ID).metadata.get("brand") == "S03-updated",
    timeout_seconds=60,
    description="metadato temporal de prueba actualizada",
)

index.delete(ids=[TEMPORARY_TEST_ID], namespace=NAMESPACE)
_, delete_visible_s, delete_attempts = wait_until(
    lambda: index.fetch(ids=[TEMPORARY_TEST_ID], namespace=NAMESPACE),
    accept=lambda response: TEMPORARY_TEST_ID not in response.vectors,
    timeout_seconds=60,
    description="borrado del registro temporal de prueba de Pinecone",
)

mutation = {"upsert": True, "fetch_or_query": True, "update": True, "delete": True}
temporary_test_visibility = {
    "upsert_seconds": upsert_visible_s,
    "upsert_attempts": upsert_attempts,
    "update_seconds": update_visible_s,
    "update_attempts": update_attempts,
    "delete_seconds": delete_visible_s,
    "delete_attempts": delete_attempts,
}
temporary_test_visibility

<a id="s03-pinecone-informe"></a>

## 8. Persistir evidencia de nuestro ejemplo con Pinecone

El informe final conservará la información necesaria para reconstruir e interpretar esta ejecución: versiones del cliente y del servicio, destino consultado, número de registros visibles, tiempos observados, scores nativos, IDs recuperados y resultado completo del canary test.

Los tiempos deben leerse como una descripción del experimento, no como un benchmark entre proveedores. No hemos controlado calentamiento, concurrencia, red, hardware ni carga de fondo, y los motores ni siquiera se ejecutan en infraestructuras comparables. Una diferencia de milisegundos no permite concluir qué solución sería más rápida en producción.

Sí podemos comparar aquello que mantuvimos constante: el contrato funcional, los filtros aplicados, la semántica de las respuestas y el `recall@10` frente al mismo oráculo exacto. Esa evidencia permite saber si cada motor respeta los datos, las condiciones y el espacio vectorial definidos para la práctica.

Persistir estos resultados no convierte una única ejecución en una verdad general. Su valor está en dejar una traza reproducible: qué se probó, bajo qué configuración y qué ocurrió exactamente.

In [ ]:
run = ProviderRun(
    provider="pinecone",
    provider_version=str(provider_version),
    target="cloud serverless aws/us-east-1",
    resource=RESOURCE_NAME,
    record_count=record_count,
    score_kind="similarity",
    higher_is_better=True,
    query_id=TELEVISOR_QUERY_ID,
    query_text=str(television_row["query_text"]),
    top_k=TOP_K,
    hits=native_hits,
    exact_record_ids=television_evaluation["exact_record_ids"],
    recall_at_k=float(television_evaluation["recall_at_k"]),
    filtered_hits=filtered_drill_hits,
    filtered_recall_at_k=float(filtered_evaluation["recall_at_k"]),
    durations_ms={"ingestion": ingestion_ms, "television_query": query_ms},
    mutation=mutation,
    visibility={"initial_count_seconds": visibility_seconds, "initial_count_attempts": visibility_attempts, **temporary_test_visibility},
    notes=["Los tiempos describen esta ejecución y no forman un ranking entre proveedores."],
)
report_path = write_provider_run(run)
print(f"Informe escrito en {report_path.relative_to(PROJECT_ROOT)}")

<a id="s03-pinecone-consideraciones"></a>

## 9. Qué es específico de Pinecone Cloud

En Pinecone, el índice serverless pertenece al plano de control y el namespace actúa como la frontera lógica de este experimento. La aplicación decide qué índice utilizar, cómo organizar los namespaces y qué metadatos enviar, pero no configura directamente el algoritmo ANN ni parámetros internos como `M`, `efSearch` o `nprobe`.

Eso significa que elegimos el contrato del servicio, no su implementación. Podemos observar la dimensión, la métrica, los filtros, la visibilidad de las escrituras y los rankings devueltos, pero no ajustar de forma explícita la estructura interna que utiliza Pinecone para recuperar candidatos. Esta diferencia debe tenerse presente al diagnosticar una pérdida de recall: algunas decisiones quedan bajo nuestro control y otras forman parte del comportamiento administrado por el proveedor.

La facilidad con la que se ejecuta el notebook cubre solo una fracción de la decisión. Antes de utilizar Pinecone en producción habría que completar una ficha operativa con evidencias sobre backups y restauración, autenticación, aislamiento entre tenants, política de upgrades, límites del servicio, observabilidad disponible, semántica de consistencia y comportamiento ante filtros muy selectivos.

También conviene distinguir qué aspectos serían costosos de cambiar una vez ingeridos millones de registros. La dimensión y la métrica forman parte de la estructura del índice y normalmente obligarían a crear otro recurso, volver a generar o cargar los vectores y migrar el tráfico. Los metadatos pueden actualizarse con mayor facilidad, pero modificar su modelo o su estrategia de filtrado también puede exigir una reingesta importante.

No todas las operaciones administrativas deberían ocultarse detrás de una interfaz aparentemente neutral. Crear un índice, elegir región o modalidad, gestionar namespaces, revisar cuotas o eliminar un recurso completo tienen consecuencias operativas específicas. Una abstracción común puede simplificar las búsquedas, pero no debería borrar decisiones que afectan a coste, seguridad o recuperación.

Antes de afirmar que el despliegue cumple un SLA, sería necesario probarlo con el volumen objetivo, concurrencia realista, filtros representativos y fallos controlados. Habría que observar no solo la latencia, sino también errores, recuperación, visibilidad de escrituras y estabilidad del recall bajo carga.

Si `recall@10` disminuye, la investigación debería comenzar por aquello que sí podemos verificar: que todos los registros sean visibles, que la consulta utilice la métrica y dimensión correctas, que el filtro no reduzca inesperadamente el universo y que el namespace sea el esperado. En un servicio donde la configuración ANN interna no se expone, la evidencia observable adquiere todavía más importancia.

## 10. Evoluciones fuera del contrato común

Cada motor ofrece capacidades que van más allá de la búsqueda densa utilizada en esta comparación: recuperación híbrida, vectores sparse, múltiples representaciones por documento, cuantización, reranking, inferencia integrada o mecanismos específicos de multitenancy.

No las incorporaremos todavía porque cambiarían la pregunta del experimento. Si un proveedor utilizara búsqueda híbrida y otro únicamente embeddings densos, una diferencia en los resultados ya no podría atribuirse con claridad al motor, al índice o a la estrategia de recuperación.

La siguiente ampliación debería comenzar siempre por un requisito medible. Por ejemplo, mejorar el recall de consultas con referencias exactas, reducir memoria o aislar tenants con una garantía concreta. A partir de ahí se añadirá una sola capacidad cada vez y se repetirá la evaluación, de modo que podamos observar qué mejora introduce y qué coste añade.

El notebook de LangChain reutilizará las colecciones ya creadas en Chroma y Qdrant. No repetirá la ingesta ni construirá una copia paralela de los datos. Si una capa de abstracción necesita duplicar toda la colección para poder conectarse, ya no estaría envolviendo el mismo sistema: estaría creando otro despliegue y alterando la comparación.

<a id="s03-pinecone-limpieza"></a>

## 11. Limpieza

Durante este laboratorio hemos creado un índice de Pinecone y hemos cargado en él los vectores del catálogo. La celda siguiente permite eliminar ese índice cuando quieras terminar la práctica. Al hacerlo desaparecerán todos sus namespaces, vectores y metadatos; no es posible deshacer la operación.

Para evitar un borrado accidental, la operación está desactivada inicialmente. Si quieres activarla, escribe en el archivo .env una confirmación con este formato:

    S03_CONFIRM_CLEANUP=DELETE:<nombre-del-índice>

Después guarda el archivo, vuelve a ejecutar la celda inicial de configuración del notebook y ejecuta la celda de limpieza. Si el valor no coincide exactamente con el índice utilizado en la práctica, el notebook no borrará nada.

Cuando termines, conviene dejar S03_CONFIRM_CLEANUP vacío de nuevo. Así, una ejecución posterior del notebook no podrá eliminar el índice por accidente.


In [ ]:
confirmation = os.getenv("S03_CONFIRM_CLEANUP", "")
expected_confirmation = f"DELETE:{RESOURCE_NAME}"

if confirmation != expected_confirmation:
    print({
        "cleanup": "omitida",
        "motivo": "Define S03_CONFIRM_CLEANUP con la confirmación exacta para habilitarla.",
        "confirmacion_requerida": expected_confirmation,
    })
else:
    client.delete_index(RESOURCE_NAME)
    print({"cleanup": "solicitada", "indice_eliminado": RESOURCE_NAME})
